# Polymarket Optimizer — Alpha Plateau Hunter

| Bucket | Mode | Strategies | What sweeps |
|--------|------|------------|-------------|
| **A** | LatencySweep | ArbYesNo, GraphArb, BtcTierArb | `network_latency_ms` ∈ [1,5,10,20,50,100] |
| **B** | CoarseGridSearch | ConvergenceNo, MeanReversion, InformedFlow, ToxicFlow, TimeDecay | 5×5 param grid at 50ms |

**Quarantined** (raises `QuarantineError`): MicroSpread, FeeArb, MarketMaking, BookImbalance.

> Opens DuckDB in **read-only** mode — no lock conflict with `run_backtest.ipynb`, no re-ingestion, no OOM.

## 0. Setup

In [1]:
import sys, pathlib, warnings
warnings.filterwarnings('ignore')

# Resolve backtest package — same logic as run_backtest.ipynb
for _p in [
    pathlib.Path.cwd().parent,
    pathlib.Path.cwd() / 'research',
    pathlib.Path.cwd(),
    pathlib.Path.cwd().parent.parent,
]:
    if (_p / 'backtest' / '__init__.py').exists():
        sys.path.insert(0, str(_p))
        break

import numpy as np
import pandas as pd
import plotly.graph_objects as go

pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', 20)

from backtest.data_loader import DataLoader
from backtest.optimizer import (
    LatencySweep, CoarseGridSearch, QuarantineError,
    LATENCY_SWEEP_VALUES, BUCKET_B_LATENCY_MS,
)
from backtest.strategies import (
    ArbYesNo, GraphArb, BtcTierArb,
    ConvergenceNo, MeanReversion, InformedFlow, ToxicFlow, TimeDecay,
)

print('Imports OK.')

Imports OK.


## 1. Load Data

Opens the DuckDB file in **`read_only=True`** mode. Multiple readers can attach simultaneously — no exclusive lock needed, no re-ingestion of 179M ticks, no OOM.

**Prerequisite:** `run_backtest.ipynb` cell 2 must have run at least once to build the `ticks` table in `/tmp/polymarket_backtest.duckdb`.

In [2]:
# Auto-detect data directory (same pattern as run_backtest.ipynb)
for _base in [
    pathlib.Path.cwd(),
    pathlib.Path.cwd().parent,
    pathlib.Path.cwd().parent.parent,
]:
    if (_base / 'data' / 'parquet').exists():
        _pq   = _base / 'data' / 'parquet'
        _meta = _base / 'data' / 'market_metadata.csv'
        break

# read_only=True: attach to the existing file without acquiring a write lock.
# _load() returns immediately in read-only mode — no staleness check, no CREATE TABLE.
loader = DataLoader(parquet_dir=_pq, metadata_csv=_meta, read_only=True)
_ = loader.con  # open connection now so the cost is visible here, not in sweep timings

s = loader.summary()
print(f"Ticks:   {s['total_ticks']:>14,}")
print(f"Trades:  {s['trades']:>14,}")
print(f"Markets: {s['markets']:>14,}")
print(f"Tokens:  {s['tokens']:>14,}")
print(f"Span:    {s['hours']:>13.1f}h")

Ticks:      516,342,067
Trades:       2,410,122
Markets:         14,196
Tokens:          28,432
Span:             60.0h


## 2. Configuration

In [3]:
STARTING_CAPITAL = 10_000   # capital gate; set to 1000 for realistic ROC

RUN_BUCKET_A = True
RUN_BUCKET_B = True

# Comment out any strategy you want to skip
BUCKET_B_SPECS = [
    dict(
        cls=ConvergenceNo,
        param1='threshold',       p1_vals=[0.25, 0.30, 0.35, 0.40, 0.45],
        param2='exit_threshold',  p2_vals=[0.75, 0.80, 0.85, 0.90, 0.95],
        fixed={'size': 10.0},
    ),
    dict(
        cls=MeanReversion,
        param1='window_s',    p1_vals=[60, 120, 300, 600, 1800],
        param2='z_threshold', p2_vals=[1.0, 1.5, 2.0, 2.5, 3.0],
        fixed={'size': 10.0},
    ),
    dict(
        cls=InformedFlow,
        param1='min_trade_size', p1_vals=[10, 25, 50, 100, 200],
        param2='window_ms',      p2_vals=[30_000, 60_000, 120_000, 300_000, 600_000],
        fixed={'size': 10.0},
    ),
    dict(
        cls=ToxicFlow,
        param1='toxicity_threshold', p1_vals=[0.55, 0.60, 0.65, 0.70, 0.75],
        param2='lookback_ms',        p2_vals=[30_000, 60_000, 120_000, 300_000, 600_000],
        fixed={'size': 10.0},
    ),
    dict(
        cls=TimeDecay,
        param1='min_hours_to_res', p1_vals=[1, 2, 4, 8, 24],
        param2='price_threshold',  p2_vals=[0.30, 0.35, 0.40, 0.45, 0.50],
        fixed={'size': 10.0},
    ),
]

print(f'Capital gate:  ${STARTING_CAPITAL:,}')
print(f'Bucket A:      {RUN_BUCKET_A}  —  latency sweep {LATENCY_SWEEP_VALUES} ms')
print(f'Bucket B:      {RUN_BUCKET_B}  —  {len(BUCKET_B_SPECS)} strategies × 25 cells at {BUCKET_B_LATENCY_MS}ms')

Capital gate:  $10,000
Bucket A:      True  —  latency sweep [1, 5, 10, 20, 50, 100] ms
Bucket B:      True  —  5 strategies × 25 cells at 50ms


---
## 3. Bucket A — Latency Sweep

**Expected:** `ArbYesNo` and `GraphArb` return 0 signals at every latency — zero arb opportunities in the ~60h dataset. `BtcTierArb` always empty (no tier metadata).

In [4]:
bucket_a_results = {}   # cls.name → DataFrame

if RUN_BUCKET_A:
    sweep = LatencySweep()
    for cls in [ArbYesNo, GraphArb, BtcTierArb]:
        try:
            df = sweep.run(
                strategy_cls=cls,
                fixed_kwargs={},
                loader=loader,
                starting_capital=STARTING_CAPITAL,
            )
            bucket_a_results[cls.name] = df
        except QuarantineError as e:
            print(f'QUARANTINED: {e}')
else:
    print('Skipped (RUN_BUCKET_A = False)')

LatencySweep: BtcTierArb: 100%|██████████| 6/6 [00:00<00:00, 216.98lat/s, fills=0, lat=100ms, pnl=$0.00]


### 3a. Results — Table + Chart

In [14]:
if not bucket_a_results:
    print('No Bucket A results.')
else:
    from IPython.display import display, HTML

    for name, df in bucket_a_results.items():
        print(f'\n  {name}')
        print(df.to_string(index=False))

    fig = go.Figure()
    for (name, df), color in zip(bucket_a_results.items(), ['#00d4aa', '#ff6b6b', '#ffd93d']):
        fig.add_trace(go.Scatter(
            x=df['latency_ms'], y=df['total_pnl'],
            mode='lines+markers', name=name,
            line=dict(color=color, width=2), marker=dict(size=8),
        ))
    fig.add_hline(y=0, line_dash='dash', line_color='gray')
    fig.update_layout(
        title='Bucket A — Total PnL vs Network Latency',
        xaxis_title='Network Latency (ms)', yaxis_title='Total PnL ($)',
        template='plotly_dark', height=400,
    )
    display(HTML(fig.to_html(full_html=False, include_plotlyjs='cdn')))


  arb_yes_no
 latency_ms  total_pnl  n_signals  n_fills  fill_rate    roc  runtime_s
          1     0.0000          0        0     0.0000 0.0000    43.8500
          5     0.0000          0        0     0.0000 0.0000    72.6500
         10     0.0000          0        0     0.0000 0.0000    63.7200
         20     0.0000          0        0     0.0000 0.0000    64.6900
         50     0.0000          0        0     0.0000 0.0000    66.2600
        100     0.0000          0        0     0.0000 0.0000    68.0700

  graph_arb
 latency_ms  total_pnl  n_signals  n_fills  fill_rate    roc  runtime_s
          1     0.0000          0        0     0.0000 0.0000    60.3200
          5     0.0000          0        0     0.0000 0.0000    67.1200
         10     0.0000          0        0     0.0000 0.0000    54.3200
         20     0.0000          0        0     0.0000 0.0000    65.8200
         50     0.0000          0        0     0.0000 0.0000    55.6600
        100     0.0000          0    

---
## 4. Bucket B — Coarse Grid Search

5×5 sweep at **50ms latency**. `tqdm` progress bars show live per-cell PnL.

**What to look for:** green clusters with ★ plateau markers — cells within 10% of global max that have ≥2 green neighbours. Isolated spikes are noise; plateaus are robust.

In [6]:
bucket_b_results = {}   # cls.name → GridResult
gs = CoarseGridSearch()

if RUN_BUCKET_B:
    for spec in BUCKET_B_SPECS:
        cls = spec['cls']
        try:
            result = gs.run(
                strategy_cls=cls,
                param1=spec['param1'],  p1_vals=spec['p1_vals'],
                param2=spec['param2'],  p2_vals=spec['p2_vals'],
                fixed_kwargs=spec['fixed'],
                loader=loader,
                starting_capital=STARTING_CAPITAL,
            )
            bucket_b_results[cls.name] = result
        except QuarantineError as e:
            print(f'QUARANTINED: {e}')
else:
    print('Skipped (RUN_BUCKET_B = False)')

GridSearch: TimeDecay: 100%|██████████| 25/25 [32:42<00:00, 78.50s/cell, p1=24, p2=0.5, pnl=$-31.06, t=79.5s] 


### 4a. Heatmaps — Total PnL

RdYlGn colorscale, zero-centred on white. ★ = plateau (robust optimum).

In [12]:
if not bucket_b_results:
    print('No Bucket B results yet — run the cell above first.')
else:
    from IPython.display import display, HTML

    for result in bucket_b_results.values():
        fig = gs.plot_heatmap(result, metric='total_pnl')
        fig.update_layout(template='plotly_dark')
        display(HTML(fig.to_html(full_html=False, include_plotlyjs='cdn')))

### 4b. Heatmaps — Return on Capital

In [10]:
if not bucket_b_results:
    print('No Bucket B results yet.')
else:
    from IPython.display import display, HTML

    for result in bucket_b_results.values():
        fig = gs.plot_heatmap(result, metric='roc')
        fig.update_layout(template='plotly_dark')
        display(HTML(fig.to_html(full_html=False, include_plotlyjs='cdn')))

---
## 5. Summary — Best Cell Per Strategy

In [11]:
if not bucket_b_results:
    print('Run Bucket B first.')
else:
    rows = []
    for name, result in bucket_b_results.items():
        z = result.pnl_matrix
        i, j = np.unravel_index(np.nanargmax(z), z.shape)
        rows.append({
            'strategy':           name,
            result.param1_name:   result.param1_values[i],
            result.param2_name:   result.param2_values[j],
            'best_pnl':           round(float(z[i, j]), 4),
            'grid_max':           round(float(np.nanmax(z)), 4),
            'grid_min':           round(float(np.nanmin(z)), 4),
            'n_positive_cells':   int((z > 0).sum()),
        })
    display(
        pd.DataFrame(rows)
          .sort_values('best_pnl', ascending=False)
          .reset_index(drop=True)
    )

,strategy,threshold,exit_threshold,best_pnl,grid_max,grid_min,n_positive_cells,window_s,z_threshold,min_trade_size,window_ms,toxicity_threshold,lookback_ms,min_hours_to_res,price_threshold
0,mean_reversion,NaN,NaN,281.1644,281.1644,219.2793,25,120.0000,1.0000,NaN,NaN,NaN,NaN,NaN,NaN
1,informed_flow,NaN,NaN,267.9758,267.9758,184.7744,25,NaN,NaN,100.0000,30000.0000,NaN,NaN,NaN,NaN
2,convergence_no,0.4000,0.9500,200.2077,200.2077,-246.4521,11,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,time_decay,NaN,NaN,132.6075,132.6075,-31.0574,4,NaN,NaN,NaN,NaN,NaN,NaN,4.0000,0.5000
4,toxic_flow,NaN,NaN,36.7505,36.7505,-186.5757,5,NaN,NaN,NaN,NaN,0.5500,600000.0000,NaN,NaN


---
## 6. Drill-Down — Zoom Into a Plateau

After spotting a plateau, edit `DRILL_P1_VALS` / `DRILL_P2_VALS` with finer values and re-run just this cell.

In [16]:
# ── Edit these ────────────────────────────────────────────────────────────
DRILL_CLS     = ConvergenceNo
DRILL_P1      = 'threshold'
DRILL_P1_VALS = [0.38, 0.40, 0.42, 0.44, 0.46]
DRILL_P2      = 'exit_threshold'
DRILL_P2_VALS = [0.90, 0.92, 0.94, 0.96, 0.98]
DRILL_FIXED   = {'size': 10.0}
# ─────────────────────────────────────────────────────────────────────────

from IPython.display import display, HTML

drill = gs.run(
    strategy_cls=DRILL_CLS,
    param1=DRILL_P1,    p1_vals=DRILL_P1_VALS,
    param2=DRILL_P2,    p2_vals=DRILL_P2_VALS,
    fixed_kwargs=DRILL_FIXED,
    loader=loader,
    starting_capital=STARTING_CAPITAL,
)

fig = gs.plot_heatmap(drill, metric='total_pnl')
fig.update_layout(title=f'{DRILL_CLS.__name__} — Drill-Down', template='plotly_dark')
display(HTML(fig.to_html(full_html=False, include_plotlyjs='cdn')))

display(drill.to_dataframe().sort_values('total_pnl', ascending=False).head(10))




GridSearch: ConvergenceNo:   0%|          | 0/25 [43:19<?, ?cell/s]
















































GridSearch: ConvergenceNo: 100%|██████████| 25/25 [3:46:27<00:00, 543.51s/cell, p1=0.46, p2=0.98, pnl=$217.69, t=245.3s]


,strategy,threshold,exit_threshold,total_pnl,roc,fill_rate,n_signals
8,convergence_no,0.4000,0.9600,226.1707,0.1625,0.9660,1061.0000
23,convergence_no,0.4600,0.9600,225.7688,0.1324,0.9550,1127.0000
9,convergence_no,0.4000,0.9800,218.0909,0.1496,0.9650,1050.0000
24,convergence_no,0.4600,0.9800,217.6890,0.1233,0.9530,1116.0000
18,convergence_no,0.4400,0.9600,201.4122,0.1292,0.9730,1105.0000
13,convergence_no,0.4200,0.9600,199.5337,0.1317,0.9690,1086.0000
19,convergence_no,0.4400,0.9800,193.2333,0.1183,0.9720,1094.0000
14,convergence_no,0.4200,0.9800,191.4538,0.1213,0.9670,1075.0000
7,convergence_no,0.4000,0.9400,191.0133,0.1387,0.9640,1067.0000
22,convergence_no,0.4600,0.9400,190.6114,0.1138,0.9530,1133.0000
